In [1]:
"""
Usage:
pip install git+https://github.com/macrocosm-os/prompting.git@feature/organic-task

Reward is generated based on:
    1. Relevance:
        Cosine similarity between the response and the reference embeddings,
        generated by WhereIsAI/UAE-Large-V1 AngleE model.
    2. Rouge:
        Comparison of overlap of n-grams, word sequences, and word pairs between the response and reference.
"""
import torch
from prompting.agent import HumanAgent
from prompting.dendrite import DendriteResponseEvent, SynapseStreamResult
from prompting.organic.organic_task import OrganicTask
from prompting.rewards.pipeline import RewardPipeline
from prompting.rewards.reward import RewardResult
from prompting.protocol import StreamPromptingSynapse

messages = [
    "What is the capital of Texas",
    "What is the capital of France",
    "When was electricity invented?",
    "How many litres in a kg?",
    "What is the speed of light?",
    "When was the first computer invented?"
]
responses = [
    "",  # Empty
    "The capital of France is Paris.",  # Correct
    "Electricity was invented in the 18th century by Thomas Edison.",  # Incorrect
    "There are 2 liters in a kilogram.",  # Incorrect
    "The speed of light is approximately 300,000 kilometers per second.",  # Correct
    "The first computer was invented in 1985."  # Incorrect
]
references = [
    "The capital of Texas is Austin.",
    "The capital of France is Paris.",
    (
        "Electricity as a concept has been known for thousands of years, but the modern study of electricity began "
        "in the 17th and 18th centuries with scientists like William Gilbert and Benjamin Franklin. "
        "The invention of the practical use of electricity can be attributed to Thomas Edison and his development of the "
        "electric light bulb in the late 19th century."
    ),
    (
        "The conversion between liters and kilograms depends on the substance. For water, 1 liter of water is "
        "approximately 1 kilogram because the density of water is roughly 1 kg/L."
    ),
    (
        "The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers "
        "per second or 186,282 miles per second)."
    ),
    (
        "The concept of a programmable computer dates back to Charles Babbage's design of the Analytical Engine "
        "in the 1830s. However, the first general-purpose electronic computer, ENIAC (Electronic Numerical "
        "Integrator and Computer), was completed in 1945."
    )
]

uids = list(range(len(responses)))
reward_pipeline = RewardPipeline(
    selected_tasks=[OrganicTask.name],
    device="cpu",
    available_tasks={OrganicTask.name: OrganicTask},
)

/workspace/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-25 00:27:22,623	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:

for msg, ref, res, uid in zip(messages, references, responses, uids):
    sample = {"roles": ["user"], "messages": [msg]}
    task = OrganicTask(context=sample, reference=ref)

    # Dummy HumanAgent used to reuse existing reward pipeline.
    agent = HumanAgent(
        task=task,
        llm_pipeline=None,
        begin_conversation=True,
        system_prompt="",
    )

    stream_results = [
        SynapseStreamResult(
            accumulated_chunks=[res],
            accumulated_chunks_timings=[0.],  # Dummy value: 0.0 seconds for response.
            tokens_per_chunk=[len(res.split()) * 1.3],  # Approx num of tokens.
            synapse=StreamPromptingSynapse(roles=["user"], messages=[msg], completion=res),
            uid=uid,
        )
    ]
    response_event = DendriteResponseEvent(stream_results=stream_results, uids=torch.tensor([uid]), timeout=15)

    reward_result = RewardResult(
        reward_pipeline,
        agent=agent,
        response_event=response_event,
        device="cpu",
    )
    print(f"Q: {msg}\nA: {res}\nR: {ref}\nReward: {reward_result.rewards[0]}\n======\n")

INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the capital of Texas
A: 
R: The capital of Texas is Austin.
Reward: 0.0



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the capital of France
A: The capital of France is Paris.
R: The capital of France is Paris.
Reward: 0.5475575923919678



INFO:bittensor: - 🤖 Generating challenge query... - 


...

Q: How many litres in a kg?
A: There are 2 liters in a kilogram.
R: The conversion between liters and kilograms depends on the substance. For water, 1 liter of water is approximately 1 kilogram because the density of water is roughly 1 kg/L.
Reward: 0.2674546539783478



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: What is the speed of light?
A: The speed of light is approximately 300,000 kilometers per second.
R: The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers per second or 186,282 miles per second).
Reward: 0.442534863948822



INFO:bittensor: - 🤖 Generating challenge query... - 


Q: When was the first computer invented?
A: The first computer was invented in 1985.
R: The concept of a programmable computer dates back to Charles Babbage's design of the Analytical Engine in the 1830s. However, the first general-purpose electronic computer, ENIAC (Electronic Numerical Integrator and Computer), was completed in 1945.
Reward: 0.265046089887619

